# Exploratory Data Analysis (EDA)

This notebook demonstrates the EDA pipeline implemented in `scripts/eda/`:
- Data cleaning (missing values, duplicates, outliers)
- Preprocessing (encoding, scaling, splitting)
- Exploratory analysis (distributions, correlations, target distribution)

In [ ]:
# Uncomment to install dependencies in Colab
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.eda.data_cleaning import report_missing, clean_pipeline
from scripts.eda.preprocessing import one_hot_encode, standard_scale, split_features_target, train_test_val_split
from scripts.eda.exploratory_analysis import describe_dataframe, plot_distributions, plot_correlation_heatmap

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load / Generate Dataset

Replace the synthetic dataset below with your own:
```python
df = pd.read_csv('../data/raw/your_dataset.csv')
```

In [ ]:
rng = np.random.default_rng(42)
n = 500
df = pd.DataFrame({
    'age':      rng.integers(18, 80, n).astype(float),
    'income':   rng.exponential(50_000, n),
    'score':    rng.normal(0.6, 0.15, n),
    'city':     rng.choice(['NY', 'LA', 'SF', 'Chicago'], n),
    'churn':    rng.integers(0, 2, n),
})
# Introduce missing values
df.loc[rng.choice(df.index, 30, replace=False), 'age'] = np.nan
df.loc[rng.choice(df.index, 10, replace=False), 'city'] = None

print(df.shape)
df.head()

## 2. Missing Value Analysis

In [ ]:
report_missing(df)

## 3. Data Cleaning Pipeline

In [ ]:
df_clean = clean_pipeline(df)
print('After cleaning:', df_clean.shape)
report_missing(df_clean)

## 4. Descriptive Statistics

In [ ]:
describe_dataframe(df_clean)

## 5. Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['age', 'income', 'score']):
    sns.histplot(df_clean[col], kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr = df_clean.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Preprocessing & Splitting

In [ ]:
df_ohe = one_hot_encode(df_clean, columns=['city'])
X, y = split_features_target(df_ohe, target_column='churn')
X_scaled, scaler = standard_scale(X)
X_train, X_val, X_test, y_train, y_val, y_test = train_test_val_split(X_scaled, y)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')